### **Tugas 1**
Terdapat dataset **mushroom**. Berdasarkan dataset yang tersebut, bandingkan peforma antara algoritma Decision Tree dan RandomForest. Gunakan tunning hyperparameter untuk mendapatkan parameter dan akurasi yang terbaik.

In [5]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Memuat dataset
data = pd.read_csv('mushrooms.csv')

# Preprocessing dataset
# Mengubah data kategorikal menjadi numerik
data = pd.get_dummies(data)

# Memisahkan fitur dan target
X = data.drop('class_e', axis=1)  # Ganti 'class_e' dengan nama kolom target yang sesuai
y = data['class_e']

# Memisahkan data menjadi set pelatihan dan pengujian
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Tugas 1 - Decision Tree
dt = DecisionTreeClassifier()
dt_param_grid = {
    'max_depth': [None, 5, 10, 15],
    'min_samples_split': [2, 5, 10]
}
dt_grid_search = GridSearchCV(dt, dt_param_grid, cv=5)
dt_grid_search.fit(X_train, y_train)

# Evaluasi Decision Tree
y_pred_dt = dt_grid_search.predict(X_test)
acc_dt = accuracy_score(y_test, y_pred_dt)
print("Best parameters for Decision Tree: ", dt_grid_search.best_params_)
print("Decision Tree Test Accuracy: {:.2f}".format(acc_dt))

# Tugas 1 - Random Forest
rf = RandomForestClassifier()
rf_param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 5, 10],
    'min_samples_split': [2, 5, 10]
}
rf_grid_search = GridSearchCV(rf, rf_param_grid, cv=5)
rf_grid_search.fit(X_train, y_train)

# Evaluasi Random Forest
y_pred_rf = rf_grid_search.predict(X_test)
acc_rf = accuracy_score(y_test, y_pred_rf)
print("Best parameters for Random Forest: ", rf_grid_search.best_params_)
print("Random Forest Test Accuracy: {:.2f}".format(acc_rf))

Best parameters for Decision Tree:  {'max_depth': None, 'min_samples_split': 2}
Decision Tree Test Accuracy: 1.00
Best parameters for Random Forest:  {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 50}
Random Forest Test Accuracy: 1.00


### **Tugas 2**
Terdapat dataset **mushroom**. Berdasarkan dataset tersebut, bandingkan peforma antara algoritma Decision Tree dan AdaBoost. Gunakan tunning hyperparameter untuk mendapatkan parameter dan akurasi yang terbaik.

In [6]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score

# Tugas 2 - Decision Tree dengan AdaBoost
# Membuat AdaBoostClassifier dengan algoritma SAMME
ada = AdaBoostClassifier(algorithm='SAMME')

# Parameter grid untuk tuning hyperparameter
ada_param_grid = {
    'n_estimators': [50, 100, 200],
    'estimator': [DecisionTreeClassifier(max_depth=depth) for depth in [1, 2, 3]]
}

# Menggunakan GridSearchCV untuk menemukan parameter terbaik
ada_grid_search = GridSearchCV(ada, ada_param_grid, cv=5, n_jobs=-1)
ada_grid_search.fit(X_train, y_train)

# Evaluasi AdaBoost
y_pred_ada = ada_grid_search.predict(X_test)
acc_ada = accuracy_score(y_test, y_pred_ada)
print("Best parameters for AdaBoost: ", ada_grid_search.best_params_)
print("AdaBoost Test Accuracy: {:.2f}".format(acc_ada))

Best parameters for AdaBoost:  {'estimator': DecisionTreeClassifier(max_depth=1), 'n_estimators': 50}
AdaBoost Test Accuracy: 1.00


### **Tugas 3**
Dengan menggunakan dataset **diabetes**, buatlah ensemble voting dengan algoritma
1. Logistic Regression
2. SVM kernel polynomial
3. Decission Tree
Anda boleh melakukan eksplorasi dengan melakukan tunning hyperparameter

In [13]:
# Import library yang dibutuhkan
import numpy as np
import pandas as pd
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Load dataset diabetes
# diabetes = load_diabetes()
dbt = pd.read_csv('diabetes.csv')


X = diabetes.data
y = diabetes.target

# Konversi target menjadi binary (0 dan 1) karena kita akan menggunakan klasifikasi
y = (y > 100).astype(int)

# Split dataset menjadi training dan testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Definisikan model-model yang akan digunakan dalam ensemble
model1 = LogisticRegression(max_iter=1000)
model2 = SVC(kernel='poly', degree=2)
model3 = DecisionTreeClassifier()

# Buat ensemble voting
ensemble = VotingClassifier(estimators=[('lr', model1), ('svm', model2), ('dt', model3)])

# Lakukan tuning hyperparameter untuk masing-masing model
param_grid = {
    'lr__C': [0.1, 1, 10],
    'svm__C': [0.1, 1, 10],
    'svm__degree': [2, 3, 4],
    'dt__max_depth': [None, 5, 10]
}

grid_search = GridSearchCV(ensemble, param_grid, cv=5, scoring='accuracy')
grid_search.fit(X_train, y_train)

# Cetak hasil tuning hyperparameter
print("Hasil Tuning Hyperparameter:")
print(grid_search.best_params_)
print("Akurasi Terbaik:", grid_search.best_score_)

# Prediksi menggunakan model terbaik
y_pred = grid_search.best_estimator_.predict(X_test)

# Evaluasi model
print("Laporan Klasifikasi:")
print(classification_report(y_test, y_pred))
print("Matriks Konfusi:")
print(confusion_matrix(y_test, y_pred))
print("Akurasi:", accuracy_score(y_test, y_pred))

Hasil Tuning Hyperparameter:
{'dt__max_depth': 5, 'lr__C': 10, 'svm__C': 1, 'svm__degree': 2}
Akurasi Terbaik: 0.7703822937625755
Laporan Klasifikasi:
              precision    recall  f1-score   support

           0       0.71      0.44      0.55        34
           1       0.72      0.89      0.80        55

    accuracy                           0.72        89
   macro avg       0.72      0.67      0.67        89
weighted avg       0.72      0.72      0.70        89

Matriks Konfusi:
[[15 19]
 [ 6 49]]
Akurasi: 0.7191011235955056
